In [ ]:
import json
import torch
import re
import time
import pandas as pd
import gc
from tqdm import tqdm
from transformers import StoppingCriteria, StoppingCriteriaList
from unsloth import FastLanguageModel

# ==========================================
# 1. FUNZIONI DI SUPPORTO E METRICHE
# ==========================================
def normalize(obj):
    """Normalizza le variabili Prolog (uppercase, _, ecc) per il confronto."""
    if not isinstance(obj, dict):
        return obj
    out = {}
    for k, v in obj.items():
        if isinstance(v, str):
            v = v.strip()
            if v == "_" or (v and v[0].isupper() and v.replace("_", "").isalpha()):
                v = "__VAR__"
        out[k] = v
    return out

def fix_hallucinated_json(text):
    """Regex attivata SOLO per il Modello Fine-Tuned. Converte '=' e protegge termini Prolog."""
    text = re.sub(r'([a-zA-Z0-9_]+)\s*=', r'"\1": ', text)
    text = re.sub(r'("arg\d+":\s*)"([^"]*)"', r'\1"\\"\2\\""', text)
    text = re.sub(r':\s*([^"\d\s\[{][^,}]*?)\s*([,}])', r': "\1"\2', text)
    return text

def calcola_accuratezza_parziale(expected_dict, predicted_dict):
    """Slot-Filling Accuracy: calcola quanti argomenti sono corretti su base parziale."""
    if not isinstance(expected_dict, dict) or not isinstance(predicted_dict, dict):
        return 0.0
    totale_chiavi = len(expected_dict)
    if totale_chiavi == 0:
        return 1.0 if len(predicted_dict) == 0 else 0.0
    chiavi_corrette = 0
    for chiave, valore in expected_dict.items():
        if chiave in predicted_dict and predicted_dict[chiave] == valore:
            chiavi_corrette += 1
    return chiavi_corrette / totale_chiavi

class StopOnTokens(StoppingCriteria):
    def __init__(self, stop_ids):
        self.stop_ids = set([s for s in stop_ids if s is not None])
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0][-1].item() in self.stop_ids

# ==========================================
# 2. CONFIGURAZIONE DEI TEST (GLI 8 FILE)
# ==========================================
esperimenti = [
    {
        "nome_modello": "Qwen_Base_4B",
        "path": "unsloth/Qwen1.5-4B-Chat-bnb-4bit", # Modello originale
        "trained": "No",
        "datasets": [
            {"id": "base_ticket_zero.jsonl", "domain": "In-Domain", "esempi": "Zero-Shot"},
            {"id": "base_ticket_few.jsonl",  "domain": "In-Domain", "esempi": "Few-Shot"},
            {"id": "base_ignoto_zero.jsonl", "domain": "Out-of-Domain", "esempi": "Zero-Shot"},
            {"id": "base_ignoto_few.jsonl",  "domain": "Out-of-Domain", "esempi": "Few-Shot"}
        ]
    },
    {
        "nome_modello": "Qwen_FineTuned_4B",
        # Sostituisci questo percorso con la tua cartella su Drive dove ci sono gli adattatori LoRA
        "path": "/content/drive/MyDrive/Tesi_BDI/modello_finetuned",
        "trained": "Yes",
        "datasets": [
            {"id": "ft_ticket_zero.jsonl", "domain": "In-Domain", "esempi": "Zero-Shot"},
            {"id": "ft_ticket_few.jsonl",  "domain": "In-Domain", "esempi": "Few-Shot"},
            {"id": "ft_ignoto_zero.jsonl", "domain": "Out-of-Domain", "esempi": "Zero-Shot"},
            {"id": "ft_ignoto_few.jsonl",  "domain": "Out-of-Domain", "esempi": "Few-Shot"}
        ]
    }
]

# ==========================================
# 3. CICLO DI INFERENZA MASSIVO
# ==========================================
risultati_globali = []

for config_mod in esperimenti:
    print(f"\n🚀 CARICAMENTO: {config_mod['nome_modello']}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = config_mod['path'],
        max_seq_length = 3072, # Lascia largo, si adatterà in automatico a 512 o 3000 reali
        dtype = None,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model)

    text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
    im_end_id = text_tokenizer.convert_tokens_to_ids("<|im_end|>")
    stop_crit = StoppingCriteriaList([StopOnTokens([text_tokenizer.eos_token_id, im_end_id])])

    for ds_config in config_mod['datasets']:
        file_name = ds_config['id']
        print(f"📊 Test in corso su: {file_name}")

        with open(file_name, encoding="utf-8") as f:
            test_cases = [json.loads(line) for line in f if line.strip()]

        for i, tc in enumerate(tqdm(test_cases, desc=f"Progresso {file_name}")):
            messages = tc["messages"]
            system   = messages[0]["content"]
            user     = messages[1]["content"]
            expected = messages[2]["content"]

            # Pialla il doppio escape Python
            user = user.replace('\\"', '"')

            prompt_testuale = text_tokenizer.apply_chat_template(
                [{"role": "system", "content": system}, {"role": "user", "content": user}],
                tokenize=False, add_generation_prompt=False
            )
            prompt_testuale += "<|im_start|>assistant\n{"

            tokens = text_tokenizer(prompt_testuale, return_tensors="pt")
            input_ids = tokens["input_ids"].to("cuda")
            attention_mask = tokens["attention_mask"].to("cuda") if "attention_mask" in tokens else torch.ones_like(input_ids)
            prompt_len = input_ids.shape[1]

            # --- START CRONOMETRO ---
            start_time = time.time()

            with torch.no_grad():
                output_ids = model.generate(
                    input_ids = input_ids,
                    attention_mask = attention_mask,
                    max_new_tokens = 128,
                    do_sample = False,
                    pad_token_id = text_tokenizer.eos_token_id,
                    stopping_criteria = stop_crit,
                )

            # --- STOP CRONOMETRO ---
            end_time = time.time()
            latenza_sec = end_time - start_time
            token_generati = len(output_ids[0]) - prompt_len
            tps = token_generati / latenza_sec if latenza_sec > 0 else 0

            predicted_raw = text_tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()
            predicted_raw = "{" + predicted_raw

            # Pulizia Markdown
            cleaned = predicted_raw
            if cleaned.startswith("
http://googleusercontent.com/immersive_entry_chip/0
http://googleusercontent.com/immersive_entry_chip/1
http://googleusercontent.com/immersive_entry_chip/2

### Come recuperare i file alla fine
Quando avrai lanciato entrambe le celle e lo script avrà scritto le varie spunte `✅`, vai nella barra di sinistra di Colab e clicca sull'icona della **Cartella**. Lì dentro vedrai spuntare `benchmark_completo_tesi.csv` e i tre file `.pdf` dei grafici. Clicca sui tre puntini di fianco ad ognuno e fai "Scarica".

Con questi strumenti in mano, la parte sperimentale della tua tesi è praticamente conclusa e pronta per essere analizzata e documentata!

In [ ]:
import json
import torch
import re
import time
import pandas as pd
import gc
import os
from tqdm import tqdm
from transformers import StoppingCriteria, StoppingCriteriaList
from unsloth import FastLanguageModel

# ==========================================
# 1. MONTAGGIO GOOGLE DRIVE E CONFIGURAZIONE PERCORSI
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

# Definiamo la cartella del tuo Drive dedicata alla tesi
# Assicurati di caricare i tuoi 8 file .jsonl dentro questa cartella su Drive!
DRIVE_FOLDER = "/content/drive/MyDrive/Tesi_BDI/"
os.makedirs(DRIVE_FOLDER, exist_ok=True) # Crea la cartella se non esiste ancora

# Percorso in cui si trovano gli adattatori LoRA del tuo modello addestrato
PATH_MODELLO_FINETUNED = os.path.join(DRIVE_FOLDER, "modello_finetuned")

# ==========================================
# 2. FUNZIONI DI SUPPORTO E METRICHE
# ==========================================
def normalize(obj):
    """Normalizza le variabili Prolog (uppercase, _, ecc) per il confronto."""
    if not isinstance(obj, dict):
        return obj
    out = {}
    for k, v in obj.items():
        if isinstance(v, str):
            v = v.strip()
            if v == "_" or (v and v[0].isupper() and v.replace("_", "").isalpha()):
                v = "__VAR__"
        out[k] = v
    return out

def fix_hallucinated_json(text):
    """Regex attivata SOLO per il Modello Fine-Tuned. Converte '=' e protegge termini Prolog."""
    text = re.sub(r'([a-zA-Z0-9_]+)\s*=', r'"\1": ', text)
    text = re.sub(r'("arg\d+":\s*)"([^"]*)"', r'\1"\\"\2\\""', text)
    text = re.sub(r':\s*([^"\d\s\[{][^,}]*?)\s*([,}])', r': "\1"\2', text)
    return text

def calcola_accuratezza_parziale(expected_dict, predicted_dict):
    """Slot-Filling Accuracy: calcola quanti argomenti sono corretti su base parziale."""
    if not isinstance(expected_dict, dict) or not isinstance(predicted_dict, dict):
        return 0.0
    totale_chiavi = len(expected_dict)
    if totale_chiavi == 0:
        return 1.0 if len(predicted_dict) == 0 else 0.0
    chiavi_corrette = 0
    for chiave, valore in expected_dict.items():
        if chiave in predicted_dict and predicted_dict[chiave] == valore:
            chiavi_corrette += 1
    return chiavi_corrette / totale_chiavi

class StopOnTokens(StoppingCriteria):
    def __init__(self, stop_ids):
        self.stop_ids = set([s for s in stop_ids if s is not None])
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0][-1].item() in self.stop_ids

# ==========================================
# 3. CONFIGURAZIONE DEI TEST (LETTI DA DRIVE)
# ==========================================
esperimenti = [
    {
        "nome_modello": "Qwen_Base_4B",
        "path": "unsloth/Qwen1.5-4B-Chat-bnb-4bit", # Scaricato al volo, non occupa spazio su Drive
        "trained": "No",
        "datasets": [
            {"id": "base_ticket_zero.jsonl", "domain": "In-Domain", "esempi": "Zero-Shot"},
            {"id": "base_ticket_few.jsonl",  "domain": "In-Domain", "esempi": "Few-Shot"},
            {"id": "base_ignoto_zero.jsonl", "domain": "Out-of-Domain", "esempi": "Zero-Shot"},
            {"id": "base_ignoto_few.jsonl",  "domain": "Out-of-Domain", "esempi": "Few-Shot"}
        ]
    },
    {
        "nome_modello": "Qwen_FineTuned_4B",
        "path": PATH_MODELLO_FINETUNED, # Carica gli adattatori direttamente da Drive
        "trained": "Yes",
        "datasets": [
            {"id": "ft_ticket_zero.jsonl", "domain": "In-Domain", "esempi": "Zero-Shot"},
            {"id": "ft_ticket_few.jsonl",  "domain": "In-Domain", "esempi": "Few-Shot"},
            {"id": "ft_ignoto_zero.jsonl", "domain": "Out-of-Domain", "esempi": "Zero-Shot"},
            {"id": "ft_ignoto_few.jsonl",  "domain": "Out-of-Domain", "esempi": "Few-Shot"}
        ]
    }
]

# ==========================================
# 4. CICLO DI INFERENZA MASSIVO
# ==========================================
risultati_globali = []

for config_mod in esperimenti:
    print(f"\n🚀 CARICAMENTO MODELLO: {config_mod['nome_modello']}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = config_mod['path'],
        max_seq_length = 3072,
        dtype = None,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model)

    text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
    im_end_id = text_tokenizer.convert_tokens_to_ids("<|im_end|>")
    stop_crit = StoppingCriteriaList([StopOnTokens([text_tokenizer.eos_token_id, im_end_id])])

    for ds_config in config_mod['datasets']:
        # Costruiamo il percorso assoluto per leggere il file dal tuo Google Drive
        file_path_drive = os.path.join(DRIVE_FOLDER, ds_config['id'])
        print(f"📊 Test in corso su: {ds_config['id']}")

        if not os.path.exists(file_path_drive):
            print(f"❌ ERRORE: Non trovo il file {file_path_drive} sul tuo Drive. Salto questo test.")
            continue

        with open(file_path_drive, encoding="utf-8") as f:
            test_cases = [json.loads(line) for line in f if line.strip()]

        for i, tc in enumerate(tqdm(test_cases, desc=f"Progresso {ds_config['id']}")):
            messages = tc["messages"]
            system   = messages[0]["content"]
            user     = messages[1]["content"]
            expected = messages[2]["content"]

            prompt_testuale = text_tokenizer.apply_chat_template(
                [{"role": "system", "content": system}, {"role": "user", "content": user}],
                tokenize=False, add_generation_prompt=False
            )
            prompt_testuale += "<|im_start|>assistant\n{"

            tokens = text_tokenizer(prompt_testuale, return_tensors="pt")
            input_ids = tokens["input_ids"].to("cuda")
            attention_mask = tokens["attention_mask"].to("cuda") if "attention_mask" in tokens else torch.ones_like(input_ids)
            prompt_len = input_ids.shape[1]

            # --- START CRONOMETRO ---
            start_time = time.time()

            with torch.no_grad():
                output_ids = model.generate(
                    input_ids = input_ids,
                    attention_mask = attention_mask,
                    max_new_tokens = 128,
                    do_sample = False,
                    pad_token_id = text_tokenizer.eos_token_id,
                    stopping_criteria = stop_crit,
                )

            # --- STOP CRONOMETRO ---
            end_time = time.time()
            latenza_sec = end_time - start_time
            token_generati = len(output_ids[0]) - prompt_len
            tps = token_generati / latenza_sec if latenza_sec > 0 else 0

            predicted_raw = text_tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()
            predicted_raw = "{" + predicted_raw

            # Pulizia Markdown
            cleaned = predicted_raw
            if cleaned.startswith("
http://googleusercontent.com/immersive_entry_chip/0
http://googleusercontent.com/immersive_entry_chip/1
http://googleusercontent.com/immersive_entry_chip/2

---





In [ ]:
### CELLA 2: Analisi Dati e Generazione Grafici PDF (Salvati Direttamente su Drive)

Questa seconda cella non legge più i file dalla memoria volatile di Colab. Carica il CSV direttamente dal tuo Google Drive, elabora le statistiche e salva i tre grafici PDF finali direttamente nella stessa cartella `Tesi_BDI` del tuo Drive. Anche se chiudi il browser, i PDF rimarranno salvati lì dentro per sempre.

```python
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Definiamo lo stesso percorso fisso del Google Drive
DRIVE_FOLDER = "/content/drive/MyDrive/Tesi_BDI/"
csv_load_path = os.path.join(DRIVE_FOLDER, "benchmark_completo_tesi.csv")

if not os.path.exists(csv_load_path):
    print(f"❌ ERRORE: Non trovo il file CSV in {csv_load_path}. Esegui prima la Cella 1!")
else:
    # 1. Carica i dati dal Drive
    df = pd.read_csv(csv_load_path)

    # Trasforma le metriche in percentuali (0-100) per i grafici
    df["Exact_Match_Pct"] = df["Exact_Match"] * 100
    df["Slot_Filling_Pct"] = df["Slot_Filling_Acc"] * 100

    # Aggregazioni Medie
    agg_df = df.groupby(["Modello", "Dominio", "Strategia"])[["Exact_Match_Pct", "Slot_Filling_Pct", "Latenza_Sec"]].mean().reset_index()

    sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

    # ==========================================
    # GRAFICO 1: Exact Match vs Slot Filling (Salva su Drive)
    # ==========================================
    ft_data = agg_df[agg_df["Modello"] == "Qwen_FineTuned_4B"].melt(
        id_vars=["Dominio", "Strategia"],
        value_vars=["Exact_Match_Pct", "Slot_Filling_Pct"],
        var_name="Metrica", value_name="Punteggio"
    )
    ft_data["Metrica"] = ft_data["Metrica"].replace({"Exact_Match_Pct": "Exact Match (100% Corretto)", "Slot_Filling_Pct": "Slot-Filling (Accuratezza Parziale)"})

    plt.figure(figsize=(10, 6))
    sns.barplot(data=ft_data, x="Strategia", y="Punteggio", hue="Metrica", palette="Set2")
    plt.title("Qwen 4B FT: Exact Match vs Slot-Filling Accuracy", fontweight="bold", pad=15)
    plt.ylabel("Accuratezza Media (%)")
    plt.xlabel("Strategia di Prompting")
    plt.ylim(0, 105)
    plt.legend(title="Metrica", loc='lower right')
    plt.tight_layout()

    plot1_path = os.path.join(DRIVE_FOLDER, "1_Metriche_Confronto.pdf")
    plt.savefig(plot1_path, dpi=300)
    plt.close()

    # ==========================================
    # GRAFICO 2: Heatmap di Generalizzazione (Salva su Drive)
    # ==========================================
    plt.figure(figsize=(8, 5))
    pivot_table = agg_df[agg_df["Modello"] == "Qwen_FineTuned_4B"].pivot_table(
        values="Slot_Filling_Pct", index="Strategia", columns="Dominio"
    )
    sns.heatmap(pivot_table, annot=True, fmt=".1f", cmap="Blues", vmin=0, vmax=100)
    plt.title("Generalizzazione (Slot-Filling %)", fontweight="bold", pad=15)
    plt.ylabel("Strategia di Prompting")
    plt.xlabel("Dominio dei Dati")
    plt.tight_layout()

    plot2_path = os.path.join(DRIVE_FOLDER, "2_Generalizzazione.pdf")
    plt.savefig(plot2_path, dpi=300)
    plt.close()

    # ==========================================
    # GRAFICO 3: Analisi Latenza (Salva su Drive)
    # ==========================================
    plt.figure(figsize=(9, 5))
    sns.barplot(data=agg_df, x="Modello", y="Latenza_Sec", hue="Strategia", palette="magma")
    plt.title("Tempo di Generazione per Query (Impatto dei Token in Input)", fontweight="bold", pad=15)
    plt.ylabel("Secondi per tradurre una frase")
    plt.xlabel("Configurazione Modello")
    plt.tight_layout()

    plot3_path = os.path.join(DRIVE_FOLDER, "3_Latenza_Confronto.pdf")
    plt.savefig(plot3_path, dpi=300)
    plt.close()

    print("✅ ANALISI COMPLETATA!")
    print(f"I seguenti grafici PDF sono stati salvati in modo permanente sul tuo Drive:")
    print(f"1. {plot1_path}")
    print(f"2. {plot2_path}")
    print(f"3. {plot3_path}")
                cleaned = cleaned.strip()

            # Il bivio: Regex applicata SOLO per il Fine-Tuned
            if config_mod['trained'] == "Yes":
                cleaned = fix_hallucinated_json(cleaned)

            # Calcolo Metriche
            passed = False
            parziale = 0.0
            try:
                dict_atteso = normalize(json.loads(expected))
                dict_previsto = normalize(json.loads(cleaned))
                passed = (dict_atteso == dict_previsto)
                parziale = calcola_accuratezza_parziale(dict_atteso, dict_previsto)
            except json.JSONDecodeError:
                pass

            risultati_globali.append({
                "Modello": config_mod['nome_modello'],
                "Addestrato": config_mod['trained'],
                "Dominio": ds_config['domain'],
                "Strategia": ds_config['esempi'],
                "Test_ID": i + 1,
                "Prompt_Tokens": prompt_len,
                "Latenza_Sec": round(latenza_sec, 3),
                "Tokens_Per_Sec": round(tps, 2),
                "Exact_Match": 1 if passed else 0,
                "Slot_Filling_Acc": round(parziale, 3),
                "Output_Grezzo": cleaned
            })

    # Svuota VRAM per il modello successivo
    print(f"🧹 Svuoto la memoria da {config_mod['nome_modello']}...")
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

# ==========================================
# 5. SALVATAGGIO PERMANENTE SU GOOGLE DRIVE
# ==========================================
df = pd.DataFrame(risultati_globali)
csv_save_path = os.path.join(DRIVE_FOLDER, "benchmark_completo_tesi.csv")
df.to_csv(csv_save_path, index=False)
print(f"\n✅ BENCHMARK COMPLETATO! Il file è salvato permanentemente sul tuo Drive in: {csv_save_path}")